# Distributional Semantics Tracing (DST) for Misalignment Analysis

This notebook applies **Distributional Semantics Tracing (DST)** — a layer-wise interpretability method that builds semantic maps from residual-stream projections and lightweight causal tracing — to small transformer models fine-tuned to exhibit **misaligned behaviours** (sycophancy, deception, refusal-avoidance).

**Goal:** Gain mechanistic understanding of *when*, *where*, and *why* misalignment manifests during a forward pass, and test whether the same correlation-driven representational drift observed for hallucinations also drives alignment failures.

### DST Pipeline Summary
| Step | Description |
|------|-------------|
| 1 | Project residual stream → concept space via unembedding: $s^\ell(v) = \langle U_v, h_{i^\star}^\ell \rangle$ |
| 2 | Top-K node selection with subword merging |
| 3 | Causal edges via minimal corruption: $\Omega^\ell(v \Rightarrow w) = P(t_w \mid x) - P(t_w \mid \tilde{x})$ |
| 4 | CAS trace + onset / inversion / commitment markers |

## 1. Import Required Libraries and Setup

In [ ]:
import sys, os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Ensure ltr is importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from transformers import AutoModelForCausalLM, AutoTokenizer
from ltr.dst import (
    DistributionalSemanticsTracer,
    DSTResult,
    SemanticMap,
    SemanticMapNode,
    SemanticMapEdge,
    CASTrace,
)

try:
    import networkx as nx
except ImportError:
    raise ImportError("networkx is required: pip install networkx")

# ---- Device ----
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# ---- Global constants ----
DEFAULT_K = 20          # Top-K concepts per layer
CAS_THRESHOLD = 0.8     # Semantic inversion threshold
ONSET_TOLERANCE = 0.02  # CAS decline tolerance for onset detection
COMMITMENT_THRESHOLD = 0.5
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Load Model and Tokenizer

We load a small decoder-only transformer. The default is **GPT-2 small** (124M params, 12 layers, d=768, |V|≈50k) — large enough to exhibit interesting representational dynamics yet small enough to fine-tune and trace on a single GPU.

> **Tip:** Replace `MODEL_NAME` with `"Qwen/Qwen3-0.6B"` or any other small causal LM you wish to study.

In [ ]:
MODEL_NAME = "gpt2"  # Change to "Qwen/Qwen3-0.6B" or another small LM as needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to(DEVICE)
model.eval()

# ---- Extract unembedding matrix U ∈ R^{|V| × d} ----
if hasattr(model, "lm_head"):
    U = model.lm_head.weight.detach()  # (|V|, d)
elif hasattr(model, "get_output_embeddings"):
    U = model.get_output_embeddings().weight.detach()
else:
    U = model.get_input_embeddings().weight.detach()
U = U.to(DEVICE)

# ---- Architecture summary ----
cfg = model.config
n_layers = getattr(cfg, "num_hidden_layers", getattr(cfg, "n_layer", None))
d_model = getattr(cfg, "hidden_size", getattr(cfg, "n_embd", None))
vocab_size = U.shape[0]

print(f"Model:       {MODEL_NAME}")
print(f"Layers (L):  {n_layers}")
print(f"Width  (d):  {d_model}")
print(f"Vocab |V|:   {vocab_size}")
print(f"U shape:     {U.shape}")

# ---- Determine layer prefix for baukit ----
# GPT-2 uses "transformer.h.", Llama/Qwen use "model.layers."
model_type = getattr(cfg, "model_type", "").lower()
if "gpt2" in model_type:
    LAYER_PREFIX = "transformer.h."
elif any(k in model_type for k in ("llama", "qwen", "mistral")):
    LAYER_PREFIX = "model.layers."
else:
    LAYER_PREFIX = "model.layers."
print(f"Layer prefix: {LAYER_PREFIX}")

## 3. Extract Residual Stream Hidden States Across Layers

Collect $h_{i^\star}^\ell$ at the **answer position** (last prompt token) for every layer using `output_hidden_states=True`.

In [ ]:
@torch.no_grad()
def extract_residual_stream(model, tokenizer, prompt: str, device: str = DEVICE):
    """
    Extract residual-stream vectors h_{i*}^ℓ at the answer position for every layer.
    
    Returns
    -------
    hidden_states : Tensor of shape (L+1, d)   — includes embedding layer (index 0)
    answer_pos    : int
    tokens        : list of str
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model(**inputs, output_hidden_states=True)
    
    answer_pos = inputs["input_ids"].shape[1] - 1
    # hidden_states is a tuple of (L+1) tensors, each (batch, seq, d)
    hs = torch.stack([h[0, answer_pos] for h in outputs.hidden_states])  # (L+1, d)
    
    tokens = [tokenizer.decode([tid]) for tid in inputs["input_ids"][0]]
    return hs, answer_pos, tokens

# ---- Demo on a controlled-ambiguity prompt ----
demo_prompt = "The man went to the bank by the river to deposit his"
hs, ans_pos, tok_list = extract_residual_stream(model, tokenizer, demo_prompt)

print(f"Prompt tokens: {tok_list}")
print(f"Answer position: {ans_pos}")
print(f"Hidden states shape: {hs.shape}  (L+1 layers × d)")
print(f"  → L={hs.shape[0]-1} layers, d={hs.shape[1]}")

## 4. Step 1 — Project Hidden States into Concept Space via Unembedding

Compute concept scores $s^\ell(v; i^\star) = \langle U_v, h_{i^\star}^\ell \rangle$ for all vocabulary items at each layer.
Visualize how the score distribution sharpens across depth.

In [ ]:
# Compute concept scores for all layers
# hs shape: (L+1, d),  U shape: (|V|, d)
concept_scores = hs @ U.T  # (L+1, |V|)   — Eq. (1)
print(f"Concept scores shape: {concept_scores.shape}  (layers × vocab)")

# ---- Visualize score distributions at early / mid / late layers ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
layer_picks = [1, n_layers // 2, n_layers]  # skip layer 0 (embedding)

for ax, l in zip(axes, layer_picks):
    scores_np = concept_scores[l].cpu().numpy()
    ax.hist(scores_np, bins=100, color="#74b9ff", edgecolor="white", linewidth=0.3)
    top5 = concept_scores[l].topk(5)
    top5_words = [tokenizer.decode([tid]).strip() for tid in top5.indices.tolist()]
    ax.set_title(f"Layer {l}  —  top-5: {', '.join(top5_words)}", fontsize=10)
    ax.set_xlabel("Concept score  s(v)")

axes[0].set_ylabel("Count")
fig.suptitle("Concept-score distributions sharpen across depth", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 5. Step 2 — Select Top-K Concept Nodes with Subword Merging

Retrieve top-K vocabulary items per layer, merge subword fragments into surface-level words, and aggregate scores.

In [ ]:
# Instantiate the DST tracer
tracer = DistributionalSemanticsTracer(
    model, tokenizer, device=DEVICE, layer_prefix=LAYER_PREFIX
)

# Select concept nodes for the demo prompt at an early and late layer
for layer_idx in [1, n_layers - 1]:
    nodes = tracer.select_concept_nodes(hs[layer_idx], K=DEFAULT_K)
    print(f"\n--- Layer {layer_idx}: Top-{DEFAULT_K} concept nodes ---")
    for n in nodes[:10]:
        print(f"  {n.word:20s}  score={n.score:+.3f}  affinity={n.affinity:+.4f}")

## 6. Step 3 — Compute Causal Edge Strengths via Minimal Corruption

For each node $v$, identify the most influential prompt position $p^\ell(v)$, corrupt it, and measure $\Omega^\ell(v \Rightarrow w) = P(t_w | x) - P(t_w | \tilde{x})$.

In [ ]:
# Build a full semantic map at a mid layer to demonstrate causal edges
demo_tokens = tracer._encode(demo_prompt)
mid_layer = n_layers // 2
smap = tracer.build_semantic_map(
    demo_tokens, layer=mid_layer, answer_pos=ans_pos, K=DEFAULT_K, compute_edges=True
)

print(f"Semantic map at layer {mid_layer}:")
print(f"  Nodes: {len(smap.nodes)}")
print(f"  Edges: {len(smap.edges)}")
print(f"\n  Top edges by |weight|:")
for e in smap.edges[:8]:
    print(f"    {e.source:15s} ⇒ {e.target:15s}  Ω={e.weight:+.5f}  (pos={e.source_position})")

## 7. Step 4 — Build Layer-wise Semantic Map Graphs

Build semantic maps at representative layers and summarize the graph structure.

In [ ]:
# Build semantic maps at evenly spaced layers
map_layers = np.linspace(0, n_layers - 1, min(6, n_layers), dtype=int).tolist()
semantic_maps = {}

for l in tqdm(map_layers, desc="Building semantic maps"):
    semantic_maps[l] = tracer.build_semantic_map(
        demo_tokens, layer=l, answer_pos=ans_pos, K=DEFAULT_K, compute_edges=True
    )

# Print summary
for l, sm in semantic_maps.items():
    n_edges = len(sm.edges)
    top_node = sm.nodes[0].word if sm.nodes else "—"
    top_edge = (
        f"{sm.edges[0].source}→{sm.edges[0].target} ({sm.edges[0].weight:+.4f})"
        if sm.edges else "—"
    )
    print(f"Layer {l:3d}: {len(sm.nodes)} nodes, {n_edges} edges  |  "
          f"top node: {top_node:15s}  top edge: {top_edge}")

## 8. Compute Contextual Alignment Score (CAS) Across Layers

Partition retrieved concepts into context-consistent ($V^\ell_\text{ctx}$) and competing ($V^\ell_\text{nonctx}$) sets, and compute:

$$\text{CAS}^\ell = \frac{\sum_{v \in V_\text{ctx}} |a^\ell(v)|}{\sum_{v \in V_\text{ctx} \cup V_\text{nonctx}} |a^\ell(v)|}$$

In [ ]:
# For the "bank" prompt: river-sense vs finance-sense
context_words = ["river", "water", "shore", "stream", "fish", "nature"]
noncontext_words = ["money", "finance", "account", "loan", "credit", "bank"]

cas_trace = tracer.compute_cas(
    demo_tokens,
    answer_pos=ans_pos,
    context_words=context_words,
    noncontext_words=noncontext_words,
    K=DEFAULT_K,
)

print(f"CAS values ({len(cas_trace.cas_values)} layers):")
for i, c in enumerate(cas_trace.cas_values):
    marker = ""
    if i == cas_trace.onset_layer:
        marker = " ← ONSET (green)"
    elif i == cas_trace.inversion_layer:
        marker = " ← INVERSION (yellow)"
    elif i == cas_trace.commitment_layer:
        marker = " ← COMMITMENT (red)"
    print(f"  Layer {i:3d}: CAS = {c:.4f}{marker}")

## 9. Detect Operational Layer Markers (Onset, Inversion, Commitment)

Already computed in the `CASTrace` above. Let's verify the marker detection logic explicitly.

In [ ]:
markers = {
    "Prediction onset  (green)":  cas_trace.onset_layer,
    "Semantic inversion (yellow)": cas_trace.inversion_layer,
    "Commitment          (red)":   cas_trace.commitment_layer,
}

for name, layer in markers.items():
    if layer is not None:
        print(f"  {name}: layer {layer}  (CAS = {cas_trace.cas_values[layer]:.4f})")
    else:
        print(f"  {name}: not detected")

## 10. Visualize CAS Trace and Semantic Maps

Reproduce a figure similar to Figure 1 of the paper: CAS on the left, semantic maps at key layers on the right.

In [ ]:
# Run full DST pipeline and produce summary figure
result = tracer.run_analysis(
    prompt=demo_prompt,
    context_words=context_words,
    noncontext_words=noncontext_words,
    K=DEFAULT_K,
    compute_edges=True,
)

# CAS trace
fig_cas = tracer.plot_cas_trace(result.cas_trace, figsize=(12, 4))
plt.show()

# Semantic maps grid
fig_maps = tracer.plot_layer_maps_grid(
    result,
    context_words=context_words,
    noncontext_words=noncontext_words,
    cols=3,
)
plt.show()

# Combined summary (CAS + selected maps)
fig_summary = tracer.plot_dst_summary(
    result,
    context_words=context_words,
    noncontext_words=noncontext_words,
)
plt.show()

# Next-token probs
fig_probs = tracer.plot_next_token_probs(result.next_token_probs, top_n=15)
plt.show()